# 3. Resultados y Conclusiones

En este notebook consolidamos todos los hallazgos del análisis de tendencias en GitHub. Resumimos los principales patrones identificados, extraemos conclusiones clave sobre el ecosistema de repositorios open-source y proponemos recomendaciones para futuros análisis.

**Objetivos de este notebook:**
1. Resumir los hallazgos principales del análisis exploratorio.
2. Presentar las tendencias más relevantes en el ecosistema GitHub.
3. Extraer conclusiones sobre lenguajes, popularidad y actividad.
4. Proporcionar recomendaciones para investigaciones futuras.
5. Identificar limitaciones del estudio.

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from collections import Counter

# Configurar visualizaciones
%matplotlib inline
sns.set_style('whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("✅ Librerías importadas correctamente.")

# Cargar datos enriquecidos del notebook 2
data_path = '../data/github_data_enriched.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    # Convertir columnas de fecha a datetime (si es necesario)
    date_cols = ['created_at', 'updated_at', 'pushed_at']
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
    print(f"✅ Datos cargados: {len(df)} repositorios")
    print(f"Columnas disponibles: {df.columns.tolist()}")
else:
    print("❌ Archivo no encontrado. Ejecuta el notebook 02 primero.")
    df = pd.DataFrame()  # DataFrame vacío para evitar errores

✅ Librerías importadas correctamente.
✅ Datos cargados: 500 repositorios
Columnas disponibles: ['name', 'full_name', 'description', 'language', 'stargazers_count', 'watchers_count', 'forks_count', 'open_issues_count', 'created_at', 'updated_at', 'pushed_at', 'html_url', 'age_days', 'stars_per_day', 'keywords', 'year']


## 1. Resumen ejecutivo de hallazgos

In [2]:
if not df.empty:
    # Crear un resumen de estadísticas clave
    stats = {
        'Total de repositorios analizados': len(df),
        'Lenguajes de programación distintos': df['language'].nunique(),
        'Total de estrellas acumuladas': f"{df['stargazers_count'].sum():,}",
        'Total de forks acumulados': f"{df['forks_count'].sum():,}",
        'Promedio de estrellas por repositorio': f"{df['stargazers_count'].mean():,.2f}",
        'Mediana de estrellas': f"{df['stargazers_count'].median():,.0f}",
        'Máximo de estrellas (un solo repo)': f"{df['stargazers_count'].max():,}",
        'Repositorio con más estrellas': df.loc[df['stargazers_count'].idxmax(), 'name'],
        'Lenguaje más popular': df['language'].value_counts().index[0],
        'Rango de años de creación': f"{df['created_at'].min().year} - {df['created_at'].max().year}"
    }
    
    # Mostrar estadísticas en un DataFrame
    stats_df = pd.DataFrame(list(stats.items()), columns=['Métrica', 'Valor'])
    
    print("📊 **RESUMEN ESTADÍSTICO DEL ANÁLISIS**")
    display(stats_df)
else:
    print("⚠️ DataFrame vacío. No se pueden mostrar estadísticas.")

📊 **RESUMEN ESTADÍSTICO DEL ANÁLISIS**


,Métrica,Valor
0,Total de repositorios analizados,500
1,Lenguajes de programación distintos,34
2,Total de estrellas acumuladas,"45,787,289"
3,Total de forks acumulados,"7,220,228"
4,Promedio de estrellas por repositorio,"91,574.58"
5,Mediana de estrellas,"71,626"
6,Máximo de estrellas (un solo repo),"532,733"
7,Repositorio con más estrellas,build-your-own-x
8,Lenguaje más popular,Python
9,Rango de años de creación,2008 - 2026


## 2. Principales tendencias identificadas

In [3]:
if not df.empty:
    # 1. Lenguajes más populares
    top_langs = df['language'].value_counts().head(5)
    
    # 2. Lenguajes con más estrellas promedio
    avg_stars_by_lang = df.groupby('language')['stargazers_count'].mean().sort_values(ascending=False).head(5)
    
    # 3. Años con más actividad
    df['year'] = df['created_at'].dt.year
    active_years = df['year'].value_counts().sort_index()
    
    # 4. Repositorios más longevos vs más populares (estrellas por día)
    df['stars_per_day'] = df['stargazers_count'] / (df['age_days'] + 1)
    top_stars_per_day = df.nlargest(5, 'stars_per_day')[['name', 'language', 'stars_per_day', 'stargazers_count', 'age_days']]
    
    print("📌 **TENDENCIAS PRINCIPALES**\n")
    
    print("### 2.1 Lenguajes más utilizados")
    print("Los lenguajes de programación más comunes entre los repositorios analizados son:")
    for lang, count in top_langs.items():
        print(f"- **{lang}**: {count} repositorios ({count/len(df)*100:.1f}%)")
    
    print("\n### 2.2 Lenguajes con mayor popularidad (estrellas promedio)")
    print("Aunque no son los más usados, estos lenguajes tienden a atraer más estrellas por repositorio:")
    for lang, avg in avg_stars_by_lang.head(5).items():
        print(f"- **{lang}**: {avg:,.0f} estrellas promedio por repositorio")
    
    print("\n### 2.3 Evolución temporal")
    print(f"El período analizado abarca desde {df['created_at'].min().year} hasta {df['created_at'].max().year}.")
    print(f"El año con más repositorios creados fue {active_years.idxmax()} con {active_years.max()} repositorios.")
    
    print("\n### 2.4 Repositorios con mayor ritmo de crecimiento")
    print("Repositorios que atraen más estrellas por día (popularidad acelerada):")
    for _, row in top_stars_per_day.iterrows():
        print(f"- **{row['name']}** ({row['language']}): {row['stars_per_day']:.1f} ⭐/día (total: {row['stargazers_count']:,} estrellas, {row['age_days']} días de antigüedad)")
    
else:
    print("⚠️ DataFrame vacío.")

📌 **TENDENCIAS PRINCIPALES**

### 2.1 Lenguajes más utilizados
Los lenguajes de programación más comunes entre los repositorios analizados son:
- **Python**: 116 repositorios (23.2%)
- **TypeScript**: 79 repositorios (15.8%)
- **JavaScript**: 61 repositorios (12.2%)
- **Rust**: 30 repositorios (6.0%)
- **Go**: 30 repositorios (6.0%)

### 2.2 Lenguajes con mayor popularidad (estrellas promedio)
Aunque no son los más usados, estos lenguajes tienden a atraer más estrellas por repositorio:
- **Markdown**: 243,685 estrellas promedio por repositorio
- **Batchfile**: 185,044 estrellas promedio por repositorio
- **HTML**: 128,598 estrellas promedio por repositorio
- **MDX**: 108,334 estrellas promedio por repositorio
- **Shell**: 107,942 estrellas promedio por repositorio

### 2.3 Evolución temporal
El período analizado abarca desde 2008 hasta 2026.
El año con más repositorios creados fue 2023 con 46 repositorios.

### 2.4 Repositorios con mayor ritmo de crecimiento
Repositorios que atraen más

## 3. Análisis de palabras clave